# 01 — Exploración de Datos iTimeControl

Este notebook explora los PDFs extraídos y verifica la calidad del preprocesamiento.

In [ ]:
import sys
sys.path.insert(0, '..')

from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from src.utils.helpers import load_config, load_json

config = load_config('../config.yaml')
print('Config cargada correctamente')

In [ ]:
# Ver PDFs disponibles
raw_dir = Path('../') / config['paths']['raw_data']
pdfs = list(raw_dir.glob('*.pdf'))
print(f'PDFs encontrados: {len(pdfs)}')
for p in pdfs:
    print(f'  - {p.name} ({p.stat().st_size / 1024:.1f} KB)')

In [ ]:
# Ver textos procesados
proc_dir = Path('../') / config['paths']['processed_data']
txts = list(proc_dir.glob('*.txt'))
print(f'Textos procesados: {len(txts)}')
for t in txts:
    content = t.read_text(encoding='utf-8')
    words = len(content.split())
    print(f'  - {t.name}: {words:,} palabras')

In [ ]:
# Ver chunks generados
chunks_dir = Path('../') / config['paths']['chunks_dir']
master = chunks_dir / 'all_chunks.json'

if master.exists():
    chunks = load_json(str(master))
    df = pd.DataFrame(chunks)
    print(f'Total chunks: {len(df)}')
    print(df[['source', 'char_count', 'word_count']].describe())
    
    # Distribución de longitudes
    fig, ax = plt.subplots(figsize=(10, 4))
    df['word_count'].hist(bins=30, ax=ax, color='steelblue', edgecolor='white')
    ax.set_xlabel('Palabras por chunk')
    ax.set_ylabel('Frecuencia')
    ax.set_title('Distribución de longitud de chunks')
    plt.tight_layout()
    plt.show()
else:
    print('Chunks no generados aún. Ejecuta el pipeline de preprocesamiento.')

In [ ]:
# Ver dataset de fine-tuning
import json
from pathlib import Path

datasets_dir = Path('../') / config['paths']['datasets_dir']

for split in ['train', 'val', 'test']:
    f = datasets_dir / f'{split}.jsonl'
    if f.exists():
        records = [json.loads(l) for l in f.read_text().splitlines() if l.strip()]
        print(f'{split}: {len(records)} registros')
        if records:
            print(f'  Ejemplo: {records[0]["instruction"][:100]}')
    else:
        print(f'{split}: no generado aún')